In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


## Importing

In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,roc_auc_score


from xgboost import XGBClassifier
%matplotlib inline

In [3]:
train=pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

print("The shape of train:",train.shape,"The shapes of test:",test.shape)

The shape of train: (594194, 21) The shapes of test: (254655, 20)


In [4]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


## Preprocessing

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254655 entries, 0 to 254654
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                254655 non-null  int64  
 1   gender            254655 non-null  object 
 2   SeniorCitizen     254655 non-null  int64  
 3   Partner           254655 non-null  object 
 4   Dependents        254655 non-null  object 
 5   tenure            254655 non-null  int64  
 6   PhoneService      254655 non-null  object 
 7   MultipleLines     254655 non-null  object 
 8   InternetService   254655 non-null  object 
 9   OnlineSecurity    254655 non-null  object 
 10  OnlineBackup      254655 non-null  object 
 11  DeviceProtection  254655 non-null  object 
 12  TechSupport       254655 non-null  object 
 13  StreamingTV       254655 non-null  object 
 14  StreamingMovies   254655 non-null  object 
 15  Contract          254655 non-null  object 
 16  PaperlessBilling  25

In [7]:
test_ids = test['id'].copy()

In [8]:
# storing IDs for submission later
train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

In [9]:
train["Churn"] = train["Churn"].map({
    "Yes":1,
    "No":0
}).astype(int)

In [10]:
y=train.pop('Churn')

In [11]:
train.head(3)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.20
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35


In [12]:
train.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.000000,594194.000000
mean,0.114102,36.577258,65.866223,2494.377057
std,0.317936,25.061922,31.067444,2353.916710
min,0.000000,1.000000,18.250000,18.800000
25%,0.000000,12.000000,29.900000,639.650000
50%,0.000000,35.000000,74.100000,1433.650000
75%,0.000000,62.000000,90.800000,4263.800000
max,1.000000,72.000000,118.750000,8684.800000


In [13]:
# checking for missing values
train.isnull().sum()


gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

In [14]:
y.value_counts()

Churn
0    460377
1    133817
Name: count, dtype: int64

In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   gender            594194 non-null  object 
 1   SeniorCitizen     594194 non-null  int64  
 2   Partner           594194 non-null  object 
 3   Dependents        594194 non-null  object 
 4   tenure            594194 non-null  int64  
 5   PhoneService      594194 non-null  object 
 6   MultipleLines     594194 non-null  object 
 7   InternetService   594194 non-null  object 
 8   OnlineSecurity    594194 non-null  object 
 9   OnlineBackup      594194 non-null  object 
 10  DeviceProtection  594194 non-null  object 
 11  TechSupport       594194 non-null  object 
 12  StreamingTV       594194 non-null  object 
 13  StreamingMovies   594194 non-null  object 
 14  Contract          594194 non-null  object 
 15  PaperlessBilling  594194 non-null  object 
 16  PaymentMethod     59

In [16]:
# printing the unique values in all the columns
numerical_features_list = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in train.columns:
  if col not in numerical_features_list:
    print(col, train[col].unique())
    print("-"*50)

gender ['Male' 'Female']
--------------------------------------------------
SeniorCitizen [0 1]
--------------------------------------------------
Partner ['Yes' 'No']
--------------------------------------------------
Dependents ['Yes' 'No']
--------------------------------------------------
PhoneService ['Yes' 'No']
--------------------------------------------------
MultipleLines ['No' 'Yes' 'No phone service']
--------------------------------------------------
InternetService ['DSL' 'Fiber optic' 'No']
--------------------------------------------------
OnlineSecurity ['Yes' 'No' 'No internet service']
--------------------------------------------------
OnlineBackup ['No' 'Yes' 'No internet service']
--------------------------------------------------
DeviceProtection ['Yes' 'No' 'No internet service']
--------------------------------------------------
TechSupport ['Yes' 'No' 'No internet service']
--------------------------------------------------
StreamingTV ['No' 'Yes' 'No internet 

## Feature Engineering

In [17]:
def apply_feature_engineering(df):
    df['promote'] = df['TotalCharges'] - (df['MonthlyCharges'] * df['tenure'])
    
    def get_rem(row):
        if row['Contract'] == 'One year':
            return max(0, 12 - row['tenure'])
        elif row['Contract'] == 'Two year':
            return max(0, 24 - row['tenure'])
        return 0
    df['remain_contract'] = df.apply(get_rem, axis=1)
    
    service_list = ["PhoneService", "MultipleLines", "OnlineSecurity", 
                    "OnlineBackup", "DeviceProtection", "TechSupport", 
                    "StreamingTV", "StreamingMovies"]

    df['total_services'] = df[service_list].isin(['Yes']).sum(axis=1)
    
    df['avg_cost_per_service'] = df['MonthlyCharges'] / df['total_services'].replace(0, 0.1)
    
    return df

In [18]:
# Apply to both datasets
train = apply_feature_engineering(train)
test = apply_feature_engineering(test)

print("Features Engineered successfully!")

Features Engineered successfully!


### Standard Scaler

In [19]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'promote', 
            'remain_contract', 'total_services', 'avg_cost_per_service']

scaler = StandardScaler()

train[num_cols] = scaler.fit_transform(train[num_cols])

test[num_cols] = scaler.transform(test[num_cols])

print("numerical features are now scaled!")

numerical features are now scaled!


In [20]:
train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,promote,remain_contract,total_services,avg_cost_per_service
0,Male,0,Yes,Yes,-0.302342,Yes,No,DSL,Yes,No,...,No,One year,Yes,Mailed check,-0.185604,-0.357076,-0.259886,-0.108892,0.246961,-0.399638
1,Male,0,Yes,Yes,0.854793,Yes,No,DSL,Yes,Yes,...,No,Two year,No,Credit card (automatic),0.116964,0.545399,-0.808008,-0.108892,0.699488,-0.446908
2,Male,0,Yes,No,0.854793,Yes,Yes,Fiber optic,No,Yes,...,Yes,Month-to-month,Yes,Electronic check,1.111575,1.421875,0.098946,-0.108892,0.699488,-0.187239
3,Female,0,No,No,-1.419575,Yes,No,Fiber optic,No,No,...,No,Month-to-month,Yes,Electronic check,0.123402,-1.029637,0.041539,-0.108892,-1.110618,1.897674
4,Female,0,No,No,-1.419575,Yes,No,Fiber optic,No,No,...,No,Month-to-month,Yes,Electronic check,0.147543,-1.029743,0.038192,-0.108892,-1.110618,1.929187


In [21]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   gender                594194 non-null  object 
 1   SeniorCitizen         594194 non-null  int64  
 2   Partner               594194 non-null  object 
 3   Dependents            594194 non-null  object 
 4   tenure                594194 non-null  float64
 5   PhoneService          594194 non-null  object 
 6   MultipleLines         594194 non-null  object 
 7   InternetService       594194 non-null  object 
 8   OnlineSecurity        594194 non-null  object 
 9   OnlineBackup          594194 non-null  object 
 10  DeviceProtection      594194 non-null  object 
 11  TechSupport           594194 non-null  object 
 12  StreamingTV           594194 non-null  object 
 13  StreamingMovies       594194 non-null  object 
 14  Contract              594194 non-null  object 
 15  

### Label Encoding

In [22]:
object_columns = train.select_dtypes(include="object").columns
encoders = {}

In [23]:
for column in object_columns:
    le = LabelEncoder()
    # for Train
    train[column] = le.fit_transform(train[column])
    
    # for test
    test[column] = le.transform(test[column])
    
    encoders[column] = le

print("Label encoded!")

Label encoded!


In [24]:
encoders

{'gender': LabelEncoder(),
 'Partner': LabelEncoder(),
 'Dependents': LabelEncoder(),
 'PhoneService': LabelEncoder(),
 'MultipleLines': LabelEncoder(),
 'InternetService': LabelEncoder(),
 'OnlineSecurity': LabelEncoder(),
 'OnlineBackup': LabelEncoder(),
 'DeviceProtection': LabelEncoder(),
 'TechSupport': LabelEncoder(),
 'StreamingTV': LabelEncoder(),
 'StreamingMovies': LabelEncoder(),
 'Contract': LabelEncoder(),
 'PaperlessBilling': LabelEncoder(),
 'PaymentMethod': LabelEncoder()}

In [25]:
train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,promote,remain_contract,total_services,avg_cost_per_service
0,1,0,1,1,-0.302342,1,0,0,2,0,...,0,1,1,3,-0.185604,-0.357076,-0.259886,-0.108892,0.246961,-0.399638
1,1,0,1,1,0.854793,1,0,0,2,2,...,0,2,0,1,0.116964,0.545399,-0.808008,-0.108892,0.699488,-0.446908
2,1,0,1,0,0.854793,1,2,1,0,2,...,2,0,1,2,1.111575,1.421875,0.098946,-0.108892,0.699488,-0.187239
3,0,0,0,0,-1.419575,1,0,1,0,0,...,0,0,1,2,0.123402,-1.029637,0.041539,-0.108892,-1.110618,1.897674
4,0,0,0,0,-1.419575,1,0,1,0,0,...,0,0,1,2,0.147543,-1.029743,0.038192,-0.108892,-1.110618,1.929187


In [26]:
train.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'promote', 'remain_contract',
       'total_services', 'avg_cost_per_service'],
      dtype='object')

### Train test split

In [27]:
# # splitting the features and target
X = train
y 

0         0
1         0
2         0
3         1
4         1
         ..
594189    0
594190    0
594191    0
594192    0
594193    1
Name: Churn, Length: 594194, dtype: int64

In [28]:
# split training and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
print("X_train shape: ", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape: ", y_train.shape)
print("y_test shape: ", y_test.shape)


X_train shape:  (475355, 23)
X_test shape:  (118839, 23)
y_train shape:  (475355,)
y_test shape:  (118839,)


In [30]:
y_train.shape


(475355,)

## Synthetic Minority Oversampling Technique (SMOTE)

In [31]:
X_train_fast = X_train.astype({col: 'float32' for col in X_train.select_dtypes('float64').columns})

In [32]:
X_train_fast = X_train_fast.astype({col: 'int32' for col in X_train_fast.select_dtypes('int64').columns})

In [33]:
smote = SMOTE(random_state=42)

In [34]:
X_train_smote, y_train_smote = smote.fit_resample(X_train_fast, y_train)

In [35]:
print("-" * 30)
print(f" SMOTE done!")
print(f"New X_train shape: {X_train_smote.shape}")
print(f"Target Distribution:\n{y_train_smote.value_counts()}")
print("-" * 30)

------------------------------
 SMOTE done!
New X_train shape: (736884, 23)
Target Distribution:
Churn
0    368442
1    368442
Name: count, dtype: int64
------------------------------


## Model Training

In [36]:
# dictionary of models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(tree_method='hist', device="cuda", random_state=42), # Use 'hist' for speed!
    "LightGBM": LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
}

In [37]:
cv_results = {}

In [38]:
warnings.filterwarnings('ignore')

for model_name, model in models.items():
    print(f" Training on : {model_name}....")

    scores = cross_val_score(model, X_train_smote, y_train_smote, cv = 5, scoring = "roc_auc", n_jobs = -1)
    cv_results[model_name] = scores
    print(f"✅ {model_name} Mean ROC-AUC: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")
    print("-" * 50)

 Training on : Decision Tree....
✅ Decision Tree Mean ROC-AUC: 0.8383 (+/- 0.0445)
--------------------------------------------------
 Training on : Random Forest....
✅ Random Forest Mean ROC-AUC: 0.9532 (+/- 0.0104)
--------------------------------------------------
 Training on : XGBoost....


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [11:49:29] WARNING: /workspace/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [11:49:29] WARNING: /workspace/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [11:49:29] WARNING: /workspace/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [11:49:29] WARNING: /workspace/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/

✅ XGBoost Mean ROC-AUC: 0.9389 (+/- 0.0151)
--------------------------------------------------
 Training on : LightGBM....
✅ LightGBM Mean ROC-AUC: 0.9557 (+/- 0.0239)
--------------------------------------------------


## Evaluations

In [39]:
# best_model = RandomForestClassifier(n_jobs=-1, random_state=42)
best_model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5)
best_model.fit(X_train_smote, y_train_smote)

rfc_probs = best_model.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'id': test_ids,
    'Churn': rfc_probs
})

submission.to_csv('submission_rfc.csv', index=False)

print("file  created!")
submission.head()

file  created!


,id,Churn
0,594194,0.144393
1,594195,0.006103
2,594196,0.341860
3,594197,0.013795
4,594198,0.780686


In [40]:
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# 2. Calculate the scores 📊
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"🎯 Test Accuracy: {accuracy:.4f}")
print(f"📈 Test ROC-AUC: {roc_auc:.4f}")
print("-" * 30)

# 3. Detailed Report (Shows Precision and Recall for Churners) 📝
print("Classification Report:")
print(classification_report(y_test, y_pred))

🎯 Test Accuracy: 0.8145
📈 Test ROC-AUC: 0.9124
------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.80      0.87     91935
           1       0.56      0.88      0.68     26904

    accuracy                           0.81    118839
   macro avg       0.76      0.84      0.78    118839
weighted avg       0.87      0.81      0.83    118839



### XGB

In [41]:
# xgb_model = XGBClassifier(tree_method='hist', random_state=42)
xgb_model = XGBClassifier(reg_alpha=0.1, reg_lambda=1.0, learning_rate=0.01)
xgb_model.fit(X_train_smote, y_train_smote)

xgb_probs = xgb_model.predict_proba(test)[:, 1]

submission_xgb = pd.DataFrame({
    'id': test_ids,
    'Churn': xgb_probs
})

submission_xgb.to_csv('submission_xgb.csv', index=False)

print(" submission_xgb.csv' created !")
submission_xgb.head()

 submission_xgb.csv' created !


,id,Churn
0,594194,0.297611
1,594195,0.186119
2,594196,0.407162
3,594197,0.196745
4,594198,0.736027


In [42]:
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"🎯 Test Accuracy: {accuracy:.4f}")
print(f"📈 Test ROC-AUC: {roc_auc:.4f}")
print("-" * 30)

print("Classification Report:")
print(classification_report(y_test, y_pred))

🎯 Test Accuracy: 0.8102
📈 Test ROC-AUC: 0.9085
------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.79      0.87     91935
           1       0.55      0.88      0.68     26904

    accuracy                           0.81    118839
   macro avg       0.75      0.83      0.77    118839
weighted avg       0.86      0.81      0.82    118839



### LGB Machine

In [43]:
lgbm_model = LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, n_jobs=-1)
lgbm_model.fit(X_train_smote, y_train_smote)

lgbm_probs = lgbm_model.predict_proba(test)[:, 1]

submission_lgbm = pd.DataFrame({'id': test_ids, 'Churn': lgbm_probs})
submission_lgbm.to_csv('submission_lgbm.csv', index=False)

print("'submission_lgbm.csv' created!")

[LightGBM] [Info] Number of positive: 368442, number of negative: 368442
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.071912 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1608
[LightGBM] [Info] Number of data points in the train set: 736884, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
'submission_lgbm.csv' created!


In [44]:
y_pred = lgbm_model.predict(X_test)
y_pred_proba = lgbm_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"🎯 Test Accuracy: {accuracy:.4f}")
print(f"📈 Test ROC-AUC: {roc_auc:.4f}")
print("-" * 30)

print("Classification Report:")
print(classification_report(y_test, y_pred))

🎯 Test Accuracy: 0.8466
📈 Test ROC-AUC: 0.9142
------------------------------
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.86      0.90     91935
           1       0.63      0.79      0.70     26904

    accuracy                           0.85    118839
   macro avg       0.78      0.83      0.80    118839
weighted avg       0.86      0.85      0.85    118839



### Ensemble results

In [45]:
final_ensemble_probs = (rfc_probs * 0.4) + (xgb_probs * 0.3) + (lgbm_probs * 0.3)

submission_final = pd.DataFrame({
    'id': test_ids,
    'Churn': final_ensemble_probs
})

submission_final.to_csv('submission_ensemble_master.csv', index=False)

print("Ensemble Submission!")
print(submission_final.head())

Ensemble Submission!
       id     Churn
0  594194  0.186200
1  594195  0.059152
2  594196  0.307134
3  594197  0.066818
4  594198  0.732439
